We'll start by scraping the web for nutritional info on various products.

In [ ]:
!pip install requests beautifulsoup4
!pip install lxml

In [1]:
import requests
from bs4 import BeautifulSoup

We'll define a separate function for each brand, and then run the function on all the relevant pages.

In [ ]:
# Beyond Meat


# function that scrapes a url for nutrition information
def beyondMeatScraper(url):

    response = requests.get(url)
    
    # if unable to fetch web page
    if response.status_code != 200:
        return None
    
    # if successful
    soup = BeautifulSoup(response.text, "html.parser")

    # compile a list of nutrients to investigate; store findings in nutrientDict
    nutrients = ["Protein", "Sodium", "Saturated Fat", "Dietary Fiber",
                 "Iron", "Vitamin B12", "Cholesterol"]
    nutrientDict = {}

    # investigate calories separately
    calories_label = soup.find("strong", string="Calories ")
    if calories_label == None:
        return None
    
    calories_value = calories_label.next_sibling
    nutrientDict.update({"Calories":calories_value})
    
    # find a tag with the td (table data) label that has a given nutrient
    for nut in nutrients:
        # finds a tag <td> such that inside is the exact string 
        nut_label = soup.find("td", string=nut)

        # gets the next <td> in the same row unless the nutrient isn't in the table
        try:
            nut_value = nut_label.find_next_sibling("td")
            nutrientDict.update({nut:nut_value.text})
        except:
            nutrientDict.update({nut:"N/A"})

    

    return nutrientDict



In [2]:
# function that gets all pages with nutrition info
def getBeyondUrls():
    response = requests.get("https://www.beyondmeat.com/sitemap.xml")
    soup = BeautifulSoup(response.text, "xml")

    # get all the URLs
    locs = soup.find_all("loc")
    all_urls = [loc.text for loc in locs]

    # get product URLs
    product_urls = [url for url in all_urls if url.startswith("https://www.beyondmeat.com/en-US/products/")]

    return product_urls



In [ ]:
getBeyondUrls()